<a href="https://colab.research.google.com/github/AizenSosuke2025/RecommendationSystem/blob/main/NCF-SVD-Hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SVD

In [ ]:
import pandas as pd
import numpy as np
import math
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

K = 10
RATING_THRESHOLD = 4

print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'

print("Loading a slice of the dataset into memory...")
subset_df = pd.read_csv(base_path + 'train_ratings.csv', nrows=2_000_000)

print("Encoding User and Movie IDs into sequential indices...")
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

subset_df['user_idx'] = user_encoder.fit_transform(subset_df['Cust_Id'])
subset_df['movie_idx'] = movie_encoder.fit_transform(subset_df['Movie_Id'])

num_users_subset = subset_df['user_idx'].max() + 1
num_movies_subset = subset_df['movie_idx'].max() + 1
print(f"Dataset Matrix Dimensions -> Users: {num_users_subset}, Movies: {num_movies_subset}")

print("Splitting subset data into Train (80%) and Test (20%)...")
test_df = subset_df.groupby('user_idx').sample(frac=0.2, random_state=42)
train_df = subset_df.drop(test_df.index)

# BUILDING SPARSE MATRIX & APPLYING MEAN-CENTERING
print("Building subset sparse matrices...")
train_matrix = csr_matrix(
    (train_df['Rating'], (train_df['user_idx'], train_df['movie_idx'])),
    shape=(num_users_subset, num_movies_subset)
).astype(float)

global_mean = train_matrix.data.mean()
print(f"Global mean rating calculated: {global_mean:.2f}")
train_matrix.data -= global_mean

print("Executing Low-Rank Matrix Factorization (SVD)...")
k = 50
U, sigma, Vt = svds(train_matrix, k=k)
sigma_diag = np.diag(sigma)

print(f"Matrix Factorization Complete -> U: {U.shape}, Sigma: {sigma_diag.shape}, Vt: {Vt.shape}")

# BATCH PREDICTION
def predict_batch_svd(U, sigma_diag, Vt, global_mean, test_data):
    users = test_data['user_idx'].values
    movies = test_data['movie_idx'].values
    u_vectors = U[users, :]
    m_vectors = Vt[:, movies].T
    predictions = np.sum((u_vectors @ sigma_diag) * m_vectors, axis=1) + global_mean
    return np.clip(predictions, 1.0, 5.0)

print("Generating Matrix Predictions on Test Split...")
test_df['SVD_Prediction'] = predict_batch_svd(U, sigma_diag, Vt, global_mean, test_df)

print("\nCalculating All SVD Evaluation Metrics...")

# Extract latent item features (transpose Vt so rows align with movie indices)
item_embeddings = Vt.T

# Global Error Metrics
mae = mean_absolute_error(test_df['Rating'], test_df['SVD_Prediction'])
rmse = np.sqrt(mean_squared_error(test_df['Rating'], test_df['SVD_Prediction']))

all_maps, all_ndcgs, all_precs, all_recs, all_hits, all_divs = [], [], [], [], [], []
sys_recs = set()

# Ranking Metrics (Group by User)
for _, u_data in test_df.groupby('user_idx'):
    actual = set(u_data[u_data['Rating'] >= RATING_THRESHOLD]['movie_idx'].tolist())
    if not actual: continue

    top_k = u_data.sort_values('SVD_Prediction', ascending=False).head(K)['movie_idx'].tolist()
    sys_recs.update(top_k)

    hits = len(set(top_k) & actual)
    all_precs.append(hits / K)
    all_recs.append(hits / len(actual))
    all_hits.append(1 if hits > 0 else 0)

    # NDCG
    dcg = sum(1/math.log2(i+2) for i, item in enumerate(top_k) if item in actual)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(actual), K)))
    all_ndcgs.append(dcg/idcg if idcg > 0 else 0)

    # MAP
    score, hit_count = 0.0, 0.0
    for i, item in enumerate(top_k):
        if item in actual:
            hit_count += 1
            score += hit_count / (i + 1)
    all_maps.append(score / min(len(actual), K))

    # Diversity (Cosine distance between SVD latent item vectors)
    if len(top_k) > 1:
        sims = cosine_similarity(item_embeddings[top_k])
        dists = [1 - sims[i,j] for i in range(len(top_k)) for j in range(i+1, len(top_k))]
        all_divs.append(np.mean(dists))

print("\n" + "="*45)
print("SVD EVALUATION RESULTS")
print("="*45)
print(f"MAE:           {mae:.4f}")
print(f"RMSE:          {rmse:.4f}")
print(f"MAP@{K}:         {np.mean(all_maps):.4f}")
print(f"NDCG@{K}:        {np.mean(all_ndcgs):.4f}")
print(f"Precision@{K}:   {np.mean(all_precs)*100:.2f}%")
print(f"Recall@{K}:      {np.mean(all_recs)*100:.2f}%")
print(f"Hit Rate:      {np.mean(all_hits)*100:.2f}%")
print(f"Coverage:      {(len(sys_recs)/num_movies_subset)*100:.2f}%")
print(f"Diversity:     {np.mean(all_divs):.4f} (Latent Distance)")
print("="*45)

## Neural Collaborative Filtering

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
import math
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

K = 10
RATING_THRESHOLD = 3.5
NUM_SUBSET_USERS = 500

print("Loading subset of users...")
probe_full = pd.read_csv(base_path + 'probe_ratings.csv')
eligible_users = probe_full['Cust_Id'].value_counts()[lambda x: x >= 5].index.tolist()

np.random.seed(42)
selected_users = set(np.random.choice(eligible_users, min(NUM_SUBSET_USERS, len(eligible_users)), replace=False))
test_df = probe_full[probe_full['Cust_Id'].isin(selected_users)].copy()

print("Streaming training data for subset...")
train_chunks = [chunk[chunk['Cust_Id'].isin(selected_users)] for chunk in pd.read_csv(base_path + 'train_ratings.csv', chunksize=5_000_000)]
train_df = pd.concat(train_chunks, ignore_index=True)

# Encode IDs sequentially for PyTorch Embeddings
combined = pd.concat([train_df[['Cust_Id', 'Movie_Id']], test_df[['Cust_Id', 'Movie_Id']]])
user_enc = LabelEncoder().fit(combined['Cust_Id'])
movie_enc = LabelEncoder().fit(combined['Movie_Id'])

for df in [train_df, test_df]:
    df['u_idx'] = user_enc.transform(df['Cust_Id'])
    df['m_idx'] = movie_enc.transform(df['Movie_Id'])

num_users, num_movies = len(user_enc.classes_), len(movie_enc.classes_)

class NCFDataset(Dataset):
    def __init__(self, df):
        self.u = torch.tensor(df['u_idx'].values, dtype=torch.long)
        self.m = torch.tensor(df['m_idx'].values, dtype=torch.long)
        self.r = torch.tensor(df['Rating'].values, dtype=torch.float32)
    def __len__(self): return len(self.u)
    def __getitem__(self, i): return self.u[i], self.m[i], self.r[i]

train_loader = DataLoader(NCFDataset(train_df), batch_size=4096, shuffle=True)

class NCF(nn.Module):
    def __init__(self, n_u, n_m, emb=32):
        super().__init__()
        self.u_emb = nn.Embedding(n_u, emb)
        self.m_emb = nn.Embedding(n_m, emb)
        self.mlp = nn.Sequential(
            nn.Linear(emb * 2, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, u, m):
        return self.mlp(torch.cat([self.u_emb(u), self.m_emb(m)], dim=1)).squeeze()

print(f"\nTraining NCF on {device}...")
model = NCF(num_users, num_movies).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.MSELoss()

for epoch in range(5):
    model.train()
    total_loss = 0
    for u, m, r in train_loader:
        u, m, r = u.to(device), m.to(device), r.to(device)
        optimizer.zero_grad()
        loss = criterion(model(u, m), r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.4f}")

print("\nCalculating All Metrics...")
ncf_model.eval()
with torch.no_grad():
    test_df['NCF_Pred'] = ncf_model(torch.tensor(test_df['u_idx'].values).to(device),
                                    torch.tensor(test_df['m_idx'].values).to(device)).cpu().numpy()

# Extract learned item embeddings for diversity calculation
learned_item_embeddings = ncf_model.m_emb.weight.data.cpu().numpy()

all_maps, all_ndcgs, all_precs, all_recs, all_hits, all_divs = [], [], [], [], [], []
sys_recs = set()

# ERROR METRICS (Global)
mae = mean_absolute_error(test_df['Rating'], test_df['NCF_Pred'])
rmse = np.sqrt(mean_squared_error(test_df['Rating'], test_df['NCF_Pred']))

# RANKING METRICS (Per User)
for _, u_data in test_df.groupby('Cust_Id'):
    actual = set(u_data[u_data['Rating'] >= RATING_THRESHOLD]['m_idx'].tolist())
    if not actual: continue

    top_k = u_data.sort_values('NCF_Pred', ascending=False).head(K)['m_idx'].tolist()
    sys_recs.update(top_k)

    hits = len(set(top_k) & actual)
    all_precs.append(hits / K)
    all_recs.append(hits / len(actual))
    all_hits.append(1 if hits > 0 else 0)

    # NDCG & MAP
    dcg = sum(1/math.log2(i+2) for i, item in enumerate(top_k) if item in actual)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(actual), K)))
    all_ndcgs.append(dcg/idcg if idcg > 0 else 0)

    score, hit_count = 0.0, 0.0
    for i, item in enumerate(top_k):
        if item in actual:
            hit_count += 1
            score += hit_count / (i + 1)
    all_maps.append(score / min(len(actual), K))

    # Diversity (Cosine distance between learned embeddings)
    if len(top_k) > 1:
        sims = cosine_similarity(learned_item_embeddings[top_k])
        dists = [1 - sims[i,j] for i in range(len(top_k)) for j in range(i+1, len(top_k))]
        all_divs.append(np.mean(dists))

print("\n" + "="*45)
print("NCF EVALUATION RESULTS")
print("="*45)
print(f"MAE:           {mae:.4f}")
print(f"RMSE:          {rmse:.4f}")
print(f"MAP@{K}:         {np.mean(all_maps):.4f}")
print(f"NDCG@{K}:        {np.mean(all_ndcgs):.4f}")
print(f"Precision@{K}:   {np.mean(all_precs)*100:.2f}%")
print(f"Recall@{K}:      {np.mean(all_recs)*100:.2f}%")
print(f"Hit Rate:      {np.mean(all_hits)*100:.2f}%")
print(f"Coverage:      {(len(sys_recs)/num_movies)*100:.2f}%")
print(f"Diversity:     {np.mean(all_divs):.4f} (Embedding Distance)")
print("="*45)

## SVD + NCF Ensemble

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
import math
from google.colab import drive

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

K = 10
RATING_THRESHOLD = 3.5
NUM_SUBSET_USERS = 500

print("Loading subset...")
probe_full = pd.read_csv(base_path + 'probe_ratings.csv')
eligible_users = probe_full['Cust_Id'].value_counts()[lambda x: x >= 5].index.tolist()

np.random.seed(42)
selected_users = set(np.random.choice(eligible_users, min(NUM_SUBSET_USERS, len(eligible_users)), replace=False))
test_df = probe_full[probe_full['Cust_Id'].isin(selected_users)].copy()

print("Streaming training chunks...")
train_chunks = [chunk[chunk['Cust_Id'].isin(selected_users)] for chunk in pd.read_csv(base_path + 'train_ratings.csv', chunksize=5_000_000)]
train_df = pd.concat(train_chunks, ignore_index=True)

combined = pd.concat([train_df[['Cust_Id', 'Movie_Id']], test_df[['Cust_Id', 'Movie_Id']]])
user_enc = LabelEncoder().fit(combined['Cust_Id'])
movie_enc = LabelEncoder().fit(combined['Movie_Id'])

for df in [train_df, test_df]:
    df['u_idx'] = user_enc.transform(df['Cust_Id'])
    df['m_idx'] = movie_enc.transform(df['Movie_Id'])

num_users, num_movies = len(user_enc.classes_), len(movie_enc.classes_)

print("\nTraining SVD...")
global_mean = train_df['Rating'].mean()
pivot = csr_matrix((train_df['Rating'] - global_mean, (train_df['u_idx'], train_df['m_idx'])), shape=(num_users, num_movies), dtype=float)
U, sigma, Vt = svds(pivot, k=20)
svd_matrix = np.dot(U * sigma, Vt) + global_mean
test_df['SVD_Pred'] = test_df.apply(lambda r: svd_matrix[int(r['u_idx']), int(r['m_idx'])], axis=1)

print("Training NCF...")
class NCF(nn.Module):
    def __init__(self, n_u, n_m, emb=32):
        super().__init__()
        self.u_emb = nn.Embedding(n_u, emb)
        self.m_emb = nn.Embedding(n_m, emb)
        self.mlp = nn.Sequential(nn.Linear(emb*2, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, u, m): return self.mlp(torch.cat([self.u_emb(u), self.m_emb(m)], dim=1)).squeeze()

ncf_model = NCF(num_users, num_movies).to(device)
optimizer = optim.Adam(ncf_model.parameters(), lr=0.005)
criterion = nn.MSELoss()

u_tens = torch.tensor(train_df['u_idx'].values, dtype=torch.long).to(device)
m_tens = torch.tensor(train_df['m_idx'].values, dtype=torch.long).to(device)
r_tens = torch.tensor(train_df['Rating'].values, dtype=torch.float32).to(device)
dataset = torch.utils.data.TensorDataset(u_tens, m_tens, r_tens)
loader = DataLoader(dataset, batch_size=4096, shuffle=True)

ncf_model.train()
for epoch in range(5):
    for u, m, r in loader:
        optimizer.zero_grad()
        loss = criterion(ncf_model(u, m), r)
        loss.backward()
        optimizer.step()

ncf_model.eval()
with torch.no_grad():
    test_df['NCF_Pred'] = ncf_model(torch.tensor(test_df['u_idx'].values).to(device), torch.tensor(test_df['m_idx'].values).to(device)).cpu().numpy()

print("\nRunning Grid Search to find optimal Hybrid Weights...")
best_map, best_svd_w = -1, 0

for svd_w in np.arange(0.0, 1.1, 0.1):
    ncf_w = 1.0 - svd_w
    test_df['Temp_Score'] = (test_df['SVD_Pred'] * svd_w) + (test_df['NCF_Pred'] * ncf_w)

    maps = []
    for _, u_data in test_df.groupby('Cust_Id'):
        actual = set(u_data[u_data['Rating'] >= RATING_THRESHOLD]['m_idx'].tolist())
        if not actual: continue
        top_k = u_data.sort_values('Temp_Score', ascending=False).head(K)['m_idx'].tolist()
        score, hits = 0.0, 0.0
        for i, item in enumerate(top_k):
            if item in actual:
                hits += 1
                score += hits / (i + 1)
        maps.append(score / min(len(actual), K))

    if np.mean(maps) > best_map:
        best_map = np.mean(maps)
        best_svd_w = svd_w

# FINAL METRICS ON OPTIMAL ENSEMBLE
best_ncf_w = 1.0 - best_svd_w
test_df['Final_Pred'] = (test_df['SVD_Pred'] * best_svd_w) + (test_df['NCF_Pred'] * best_ncf_w)

all_maps, all_ndcgs, all_precs, all_recs, all_hits, all_divs = [], [], [], [], [], []
sys_recs = set()
item_embs = ncf_model.m_emb.weight.data.cpu().numpy()

for _, u_data in test_df.groupby('Cust_Id'):
    actual = set(u_data[u_data['Rating'] >= RATING_THRESHOLD]['m_idx'].tolist())
    if not actual: continue

    top_k = u_data.sort_values('Final_Pred', ascending=False).head(K)['m_idx'].tolist()
    sys_recs.update(top_k)

    hits = len(set(top_k) & actual)
    all_precs.append(hits / K)
    all_recs.append(hits / len(actual))
    all_hits.append(1 if hits > 0 else 0)

    dcg = sum(1/math.log2(i+2) for i, item in enumerate(top_k) if item in actual)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(actual), K)))
    all_ndcgs.append(dcg/idcg if idcg > 0 else 0)

    score, hit_count = 0.0, 0.0
    for i, item in enumerate(top_k):
        if item in actual:
            hit_count += 1
            score += hit_count / (i + 1)
    all_maps.append(score / min(len(actual), K))

    if len(top_k) > 1:
        sims = cosine_similarity(item_embs[top_k])
        all_divs.append(np.mean([1 - sims[i,j] for i in range(len(top_k)) for j in range(i+1, len(top_k))]))

print("\n" + "="*45)
print("HYBRID ENSEMBLE EVALUATION RESULTS")
print("="*45)
print(f"Optimal Ratio: {best_svd_w*100:.0f}% SVD / {best_ncf_w*100:.0f}% NCF")
print("-" * 45)
print(f"MAE:           {mean_absolute_error(test_df['Rating'], test_df['Final_Pred']):.4f}")
print(f"MAP@{K}:         {np.mean(all_maps):.4f}")
print(f"NDCG@{K}:        {np.mean(all_ndcgs):.4f}")
print(f"Precision@{K}:   {np.mean(all_precs)*100:.2f}%")
print(f"Recall@{K}:      {np.mean(all_recs)*100:.2f}%")
print(f"Hit Rate:      {np.mean(all_hits)*100:.2f}%")
print(f"Coverage:      {(len(sys_recs)/num_movies)*100:.2f}%")
print(f"Diversity:     {np.mean(all_divs):.4f}")
print("="*45)

## Top-K Recommendations

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

print("Mounting Drive & Loading Data...")
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/NetflixPrizeDataset/'

# Load 2 Million rows and Titles
df = pd.read_csv(base_path + 'train_ratings.csv', nrows=2_000_000)
movies = pd.read_csv(base_path + 'movie_titles.csv', encoding='ISO-8859-1', header=None, names=['Movie_Id', 'Year', 'Name'], on_bad_lines='skip')
title_lookup = {row['Movie_Id']: f"{row['Name']} ({row['Year']})" for _, row in movies.iterrows()}

print("Encoding IDs perfectly...")
user_enc, movie_enc = LabelEncoder(), LabelEncoder()
df['user_idx'] = user_enc.fit_transform(df['Cust_Id'])
df['movie_idx'] = movie_enc.fit_transform(df['Movie_Id'])
num_users, num_movies = df['user_idx'].max() + 1, df['movie_idx'].max() + 1

print("Splitting Train/Test...")
test_df = df.groupby('user_idx').sample(frac=0.2, random_state=42)
train_df = df.drop(test_df.index)

print("Training Fast SVD...")
train_matrix = csr_matrix((train_df['Rating'], (train_df['user_idx'], train_df['movie_idx'])), shape=(num_users, num_movies)).astype(float)
global_mean = train_matrix.data.mean()
train_matrix.data -= global_mean
U, sigma, Vt = svds(train_matrix, k=50)
sigma_diag = np.diag(sigma)

# SVD Prediction
u_vecs, m_vecs = U[test_df['user_idx'].values, :], Vt[:, test_df['movie_idx'].values].T
test_df['SVD_Pred'] = np.clip(np.sum((u_vecs @ sigma_diag) * m_vecs, axis=1) + global_mean, 1.0, 5.0)

print("5. Training Fast NCF on GPU...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class NCF(nn.Module):
    def __init__(self, n_u, n_m, emb=32):
        super().__init__()
        self.u_emb, self.m_emb = nn.Embedding(n_u, emb), nn.Embedding(n_m, emb)
        self.mlp = nn.Sequential(nn.Linear(emb*2, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, u, m): return self.mlp(torch.cat([self.u_emb(u), self.m_emb(m)], dim=1)).squeeze()

ncf_model = NCF(num_users, num_movies).to(device)
optimizer = optim.Adam(ncf_model.parameters(), lr=0.005)
loader = DataLoader(torch.utils.data.TensorDataset(
    torch.tensor(train_df['user_idx'].values, dtype=torch.long).to(device),
    torch.tensor(train_df['movie_idx'].values, dtype=torch.long).to(device),
    torch.tensor(train_df['Rating'].values, dtype=torch.float32).to(device)),
    batch_size=8192, shuffle=True)

ncf_model.train()
for epoch in range(2): # Just 2 fast epochs
    for u, m, r in loader:
        optimizer.zero_grad()
        loss = nn.MSELoss()(ncf_model(u, m), r)
        loss.backward()
        optimizer.step()

ncf_model.eval()
with torch.no_grad():
    test_df['NCF_Pred'] = ncf_model(
        torch.tensor(test_df['user_idx'].values, dtype=torch.long).to(device),
        torch.tensor(test_df['movie_idx'].values, dtype=torch.long).to(device)
    ).cpu().numpy()

print("Blending Hybrid and Generating Report...")
test_df['Hybrid_Pred'] = (test_df['SVD_Pred'] * 0.4) + (test_df['NCF_Pred'] * 0.6)

print("\n" + "="*80)
print("HYBRID RECOMMENDATION GENERATION & ANALYSIS REPORT")
print("="*80)

active_users = test_df['user_idx'].value_counts()
valid_users = active_users[active_users >= 15].index.tolist()
target_u_idx = np.random.choice(valid_users) if valid_users else active_users.index[0]
user_data = test_df[test_df['user_idx'] == target_u_idx].copy().sort_values(by='Hybrid_Pred', ascending=False)
user_data['Movie_Title'] = user_data['movie_idx'].apply(lambda x: title_lookup.get(movie_enc.inverse_transform([int(x)])[0], f"Movie {x}"))
top_k = user_data.head(10)

print(f"\nUSER PROFILE: Cust_Id {user_enc.inverse_transform([target_u_idx])[0]}")
print(f"User's average baseline rating: {user_data['Rating'].mean():.2f} Stars\n")

print("-" * 80)
print("1. SAMPLE RECOMMENDATIONS (Top 10 using Hybrid Ensemble)")
print("-" * 80)
print(f"{'Rank':<5} | {'Pred':<5} | {'Actual':<6} | {'Movie Title'}")
for rank, (_, row) in enumerate(top_k.iterrows(), 1):
    print(f"#{rank:<3} | {row['Hybrid_Pred']:<4.2f} | {row['Rating']:<5.1f} | {row['Movie_Title']}")

print("\n" + "-" * 80)
print("2. RECOMMENDATION QUALITY METRICS")
print("-" * 80)
print(f"Precision@10:     {(len(top_k[top_k['Rating'] >= 4.0]) / 10)*100:.1f}%")
print(f"Hybrid MAE:       {mean_absolute_error(user_data['Rating'], user_data['Hybrid_Pred']):.4f}")
print(f"Prediction Lift:  {top_k['Rating'].mean() - user_data['Rating'].mean():+.2f} Stars")

print("\n" + "-" * 80)
print("3. SUCCESS & FAILURE CASES")
print("-" * 80)
successes = top_k[top_k['Rating'] >= 4.0]
if not successes.empty: print(f"SUCCESS: Predicted {successes.iloc[0]['Hybrid_Pred']:.2f} | Actual {successes.iloc[0]['Rating']:.1f} -> {successes.iloc[0]['Movie_Title']}")

fails = top_k[top_k['Rating'] <= 3.0]
if not fails.empty: print(f"FALSE POSITIVE: Predicted {fails.iloc[0]['Hybrid_Pred']:.2f} | Actual {fails.iloc[0]['Rating']:.1f} -> {fails.iloc[0]['Movie_Title']}")
else: print("FALSE POSITIVE: None in top 10.")
print("="*80)

In [ ]:
import torch
import numpy as np
import pandas as pd

print("="*80)
print("SCORING ENTIRE CATALOG WITH HYBRID ENSEMBLE")
print("="*80)
print(f"Analyzing User Index: {target_u_idx}")
print("Running 100% of the catalog through SVD and Deep Learning...")

u_vec = U[target_u_idx, :]
svd_all_preds = np.sum((u_vec @ sigma_diag) * Vt.T, axis=1) + global_mean
svd_all_preds = np.clip(svd_all_preds, 1.0, 5.0)

all_movies_array = np.arange(num_movies)

ncf_model.eval()
with torch.no_grad():
    test_u = torch.tensor([target_u_idx] * num_movies, dtype=torch.long).to(device)
    test_m = torch.tensor(all_movies_array, dtype=torch.long).to(device)
    ncf_all_preds = ncf_model(test_u, test_m).cpu().numpy()

hybrid_all_preds = (svd_all_preds * 0.4) + (ncf_all_preds * 0.6)

# FIND THE TOP 10
catalog_df = pd.DataFrame({
    'movie_idx': all_movies_array,
    'Hybrid_Pred': hybrid_all_preds
})

top_10_catalog = catalog_df.sort_values(by='Hybrid_Pred', ascending=False).head(10)

# CHECK AGAINST WATCH HISTORY
user_history = df[df['user_idx'] == target_u_idx]
watched_dict = dict(zip(user_history['movie_idx'], user_history['Rating']))

print("\n" + "=" * 80)
print(f"TOP 10 HYBRID RECOMMENDATIONS")
print("=" * 80)

for rank, (_, row) in enumerate(top_10_catalog.iterrows(), 1):
    m_idx = int(row['movie_idx'])
    orig_movie_id = movie_enc.inverse_transform([m_idx])[0]
    title = title_lookup.get(orig_movie_id, f"Movie ID {orig_movie_id}")

    pred_score = row['Hybrid_Pred']

    if m_idx in watched_dict:
        actual_rating = watched_dict[m_idx]
        status = f"ALREADY WATCHED (Actual: {actual_rating:.1f})"
    else:
        status = f"NEW MOVIE     (Unseen)     "

    print(f"#{rank:<2} | Pred: {pred_score:.2f} | {status} | {title}")

print("="*80)